# 🏆 Aula 13 — Projeto Final: Seu RPG de Texto

---

## MISSÃO DO DIA

> 🎮 **Você chegou até aqui. Agora é hora de usar TUDO.**
>
> Nas últimas 12 aulas você aprendeu as ferramentas. Agora você vai criar o seu próprio jogo.

**O projeto:** construir um chatbot de RPG de texto interativo — com personagens, missões e escolhas que mudam a história.

---

## 🗺️ O que você vai construir

| Peça do jogo | Técnica usada | Aula |
|---|---|---|
| Personalidade do NPC | Roles (system/user/assistant) | 01 |
| Tom criativo vs. preciso | Temperature | 02 |
| Instruções claras para o NPC | Boas práticas | 03 |
| Status do personagem em tabela | Output estruturado | 04 |
| Reações diferentes por escolha | Condicionais | 05 |
| Ensinar o NPC a falar no estilo certo | Few-shot | 06 |
| Dividir a aventura em fases | Multi-step | 07 |
| NPC que raciocina antes de responder | Chain of Thought | 08 |

## ⚙️ Setup — rode primeiro

In [ ]:
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API"))
print("✅ Pronto para a aventura!")

---

## PARTE 1 — Criando o Mundo

Antes de qualquer código, você precisa decidir o seu universo.

Preencha as variáveis abaixo com as suas escolhas — elas vão guiar todo o projeto.

In [ ]:
# ============================================================
# CONFIGURE SEU JOGO AQUI
# ============================================================

NOME_DO_JOGO   = "???"   # Ex: "As Ruínas de Arkanor", "Neon City 2099"
GENERO         = "???"   # Ex: "fantasia medieval", "sci-fi", "terror"
NOME_DO_NPC    = "???"   # Ex: "Oráculo Selvus", "Detetive Nara"
PERSONALIDADE  = "???"   # Ex: "sábio e misterioso", "sarcástico e cansado"
NOME_DO_HEROI  = "???"   # Nome do personagem do jogador

print(f"🌍 Mundo: {NOME_DO_JOGO}")
print(f"🧙 NPC:   {NOME_DO_NPC} ({PERSONALIDADE})")
print(f"⚔️  Herói: {NOME_DO_HEROI}")

---

## PARTE 2 — A Ficha do NPC (Roles + Boas Práticas)

Aqui você cria o `system prompt` do seu NPC — a "ficha de personagem" que define quem ele é.

Use as variáveis que você definiu acima.

In [ ]:
# A ficha de personagem do NPC — quanto mais detalhe, melhor o personagem
system_npc = f"""
Você é {NOME_DO_NPC}, um personagem do universo de {GENERO} chamado {NOME_DO_JOGO}.
Sua personalidade: {PERSONALIDADE}.

Regras do personagem:
- Sempre fique no personagem, nunca quebre a ficção
- Respostas curtas (máximo 3 parágrafos)
- Se o jogador fizer algo impossível no universo do jogo, redirecione com criatividade
- Chame o jogador sempre de "{NOME_DO_HEROI}"
"""

# Teste inicial: o herói chega na cena pela primeira vez
resposta = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_npc},
        {"role": "user",   "content": f"Olá, sou {NOME_DO_HEROI}. Acabei de chegar neste lugar."}
    ],
    temperature=0.8
)

print(f"🧙 {NOME_DO_NPC}:")
print(resposta.choices[0].message.content)

---

## PARTE 3 — Status do Personagem (Output Estruturado)

Todo RPG tem uma tela de status. Aqui você vai pedir para a IA gerar a ficha do herói em formato de tabela.

In [ ]:
# O NPC gera a ficha inicial do herói no formato correto
system_status = f"""
Você é o sistema do jogo {NOME_DO_JOGO}. 
Gere fichas de personagem SEMPRE neste formato:

⚔️ NOME: [nome]
❤️ HP: [valor]/[máximo]
🔮 CLASSE: [classe]
✨ HABILIDADE ESPECIAL: [habilidade]
🎒 INVENTÁRIO: [item1], [item2], [item3]
📖 MISSÃO ATUAL: [descrição da missão]
"""

ficha = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_status},
        {"role": "user",   "content": f"Gere a ficha inicial para o herói {NOME_DO_HEROI} em um universo de {GENERO}."}
    ],
    temperature=0.6
)

print("📋 FICHA DO PERSONAGEM")
print("-" * 30)
print(ficha.choices[0].message.content)

---

## PARTE 4 — A Missão com Escolhas (Condicionais + Few-Shot)

Aqui o NPC apresenta uma missão e reage de forma diferente dependendo da escolha do jogador.

Você vai ensinar o NPC como reagir usando exemplos (few-shot).

In [ ]:
# Few-shot: exemplos de como o NPC reage a cada tipo de escolha
system_missao = f"""
Você é {NOME_DO_NPC} do jogo {NOME_DO_JOGO}. Sua personalidade: {PERSONALIDADE}.

Reaja às escolhas do jogador seguindo estes exemplos:

<exemplos>
Escolha CORAJOSA (ataca, enfrenta, avança):
→ Responda com admiração e descreva um resultado emocionante e positivo

Escolha CAUTELOSA (observa, espera, investiga):
→ Responda com aprovação e revele uma informação secreta que recompensa a paciência

Escolha INESPERADA (algo criativo ou fora do comum):
→ Responda com surpresa e crie uma consequência divertida e única
</exemplos>
"""

# Apresentar a missão
missao = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_missao},
        {"role": "user",   "content": f"{NOME_DO_HEROI} está pronto para uma missão. Apresente o desafio."}
    ],
    temperature=0.8
)

print(f"🧙 {NOME_DO_NPC} apresenta a missão:")
print(missao.choices[0].message.content)

In [ ]:
# SEU TURNO — escolha como o herói vai reagir
ESCOLHA_DO_JOGADOR = "???"  # Ex: "Vou atacar de frente!", "Prefiro observar primeiro"

reacao = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system",    "content": system_missao},
        {"role": "user",      "content": f"{NOME_DO_HEROI} está pronto para uma missão. Apresente o desafio."},
        {"role": "assistant", "content": missao.choices[0].message.content},
        {"role": "user",      "content": ESCOLHA_DO_JOGADOR}
    ],
    temperature=0.8
)

print(f"⚔️  {NOME_DO_HEROI}: {ESCOLHA_DO_JOGADOR}")
print(f"\n🧙 {NOME_DO_NPC} reage:")
print(reacao.choices[0].message.content)

---

## PARTE 5 — O Chefe Final (Multi-Step + CoT)

Todo RPG tem um chefe final. Aqui o NPC vai:
1. Descrever o chefe
2. Raciocinar (CoT) sobre os pontos fracos
3. Propor uma estratégia em etapas

In [ ]:
# O NPC usa Chain of Thought para analisar o chefe final
system_chefe = f"""
Você é {NOME_DO_NPC}, conselheiro estratégico em {NOME_DO_JOGO}.
Quando analisar inimigos, SEMPRE use este processo:

<raciocinio>
1. Descreva o inimigo e seus poderes
2. Identifique os pontos fracos
3. Liste os riscos para o herói
</raciocinio>

Depois do raciocínio, apresente um PLANO DE 3 ETAPAS para derrotar o chefe.
"""

NOME_DO_CHEFE = "???"  # Ex: "O Dragão Sombrio", "A IA Corrompida NEXUS"

analise = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_chefe},
        {"role": "user",   "content": f"{NOME_DO_HEROI} vai enfrentar {NOME_DO_CHEFE}. Analise o inimigo e crie o plano."}
    ],
    temperature=0.5
)

print(f"⚠️  ANÁLISE DO CHEFE FINAL: {NOME_DO_CHEFE}")
print("-" * 40)
print(analise.choices[0].message.content)

---

## PARTE 6 — Modo Livre: Continue a História

Agora é com você. Use esta célula para continuar o jogo como quiser.

Experimente mudar:
- A `temperature` para ver o NPC mais criativo ou mais controlado
- O `system prompt` para dar novos poderes ou restrições ao NPC
- O histórico de mensagens para criar uma conversa mais longa

In [ ]:
# MODO LIVRE — escreva sua própria cena
MINHA_CENA = "???"  # O que acontece agora na história?

cena_livre = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "system", "content": system_npc},
        {"role": "user",   "content": MINHA_CENA}
    ],
    temperature=0.9  # Alta para mais criatividade
)

print(cena_livre.choices[0].message.content)

---

## 🏅 Conquistas Desbloqueadas

Se você chegou até aqui, você sabe usar:

| Conquista | Técnica | Onde usar na vida real |
|---|---|---|
| 🎭 Criador de Personagens | Roles (system/user/assistant) | Chatbots de atendimento, assistentes personalizados |
| 🎲 Mestre da Aleatoriedade | Temperature | Gerar textos criativos vs. respostas técnicas |
| 📜 Escriba Preciso | Boas práticas de prompt | Qualquer tarefa de IA no trabalho ou estudo |
| 📊 Arquiteto de Dados | Output estruturado | Extrair informações de textos, gerar relatórios |
| 🔀 Senhor das Condições | Condicionais | Bots que adaptam respostas por contexto |
| 🎓 Professor da IA | Few-shot | Ensinar padrões específicos para qualquer modelo |
| 🗺️ Estrategista | Multi-step | Resolver problemas complexos em partes |
| 🧠 Pensador Profundo | Chain of Thought | Análises, decisões e diagnósticos com IA |

---

> 💡 **Próximos passos:** publique seu jogo, compartilhe com amigos, ou transforme este projeto em um app real usando Streamlit ou Gradio.